# 15 — E0 Pair Selection (Step 8)

Selects the concept pairs for the E0 pilot: compute ρ over ~30 regime-tagged candidate
pairs, then pick three at **low / medium / high ρ** (preferring synsets rich in
multi-token lemmas, avoiding hub concepts).

Each pair carries two annotations that E0's analysis needs (see
`plans/multiconcept-cgd-implementation-plan.md`, Step 8):

- **semantic regime** (similar / antonym / unrelated / compositional / hypernym) —
  ρ strata are assigned by *measured* ρ; if regime and ρ dissociate (antonyms are
  expected to be geometrically similar), that is a finding, not noise
- **word-level co-realizability** (manual annotation): can both concepts be expressed
  in one word? Pairs that can't are where Family 3 is the control condition

Outputs (new files, baseline `resources/` untouched):
- `resources/15_e0_rho_by_regime.png`
- `resources/15_e0_selected_pairs.json`

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.representations import FrameUnembeddingRepresentation

In [ ]:
fur = FrameUnembeddingRepresentation.from_model_id(
    "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

MIN_LEMMAS = 11
MAX_TOKENS = 3

## Candidate pairs, tagged by regime

`word_level=True` means both concepts could plausibly co-occur in a single word
(manual judgment, recorded for the Family-3 control analysis).

In [ ]:
PAIRS = [
    # (a, b, regime, word_level)
    ("joy.n.01", "happiness.n.01", "similar", True),
    ("sorrow.n.01", "sadness.n.01", "similar", True),
    ("dog.n.01", "cat.n.01", "similar", False),
    ("king.n.01", "queen.n.02", "similar", False),
    ("mother.n.01", "father.n.01", "similar", False),
    ("joy.n.01", "sorrow.n.01", "antonym", False),
    ("happiness.n.01", "sadness.n.01", "antonym", False),
    ("love.n.01", "hate.n.01", "antonym", True),
    ("war.n.01", "peace.n.01", "antonym", False),
    ("black.n.01", "white.n.01", "antonym", False),
    ("fire.n.01", "water.n.01", "antonym", False),
    ("dog.n.01", "mathematics.n.01", "unrelated", False),
    ("music.n.01", "water.n.01", "unrelated", False),
    ("sadness.n.01", "music.n.01", "unrelated", True),
    ("joy.n.01", "music.n.01", "unrelated", True),
    ("cat.n.01", "mathematics.n.01", "unrelated", False),
    ("war.n.01", "water.n.01", "unrelated", False),
    ("snow.n.01", "fire.n.01", "unrelated", False),
    ("joy.n.01", "dog.n.01", "unrelated", False),
    ("woman.n.01", "child.n.01", "compositional", True),  # -> girl.n.01
    ("woman.n.01", "king.n.01", "compositional", True),  # -> queen.n.02
    ("storm.n.01", "snow.n.01", "compositional", True),  # -> blizzard-ish
    ("dog.n.01", "animal.n.01", "hypernym", True),
    ("puppy.n.01", "dog.n.01", "hypernym", True),
    ("cat.n.01", "animal.n.01", "hypernym", True),
    ("girl.n.01", "woman.n.01", "hypernym", True),
    ("girl.n.01", "child.n.01", "hypernym", True),
    ("queen.n.02", "woman.n.01", "hypernym", True),
    ("puppy.n.01", "animal.n.01", "hypernym", True),
    ("storm.n.01", "water.n.01", "related", True),
]

COMPOSITION_TARGETS = {
    ("woman.n.01", "child.n.01"): "girl.n.01",
    ("woman.n.01", "king.n.01"): "queen.n.02",
}

HUB_CONCEPTS = {"child.n.01"}  # correlates with everything (step 3 finding)

len(PAIRS)

## Survival + multi-token richness per synset

In [ ]:
df = fur.data.get_dataframe(fur.model.tokenizer, MIN_LEMMAS, MAX_TOKENS)
df = df.to_pandas() if hasattr(df, "to_pandas") else df

synsets = sorted({s for a, b, *_ in PAIRS for s in (a, b)})
stats = []
for synset in synsets:
    sub = df[df["synset"] == synset]
    stats.append(
        {
            "synset": synset,
            "available": len(sub) > 0,
            "lemmas": len(sub),
            "multi_token_share": (
                float((sub["tokens"].apply(len) > 1).mean()) if len(sub) else 0.0
            ),
        }
    )
stats = pd.DataFrame(stats).set_index("synset")
missing = stats[~stats["available"]].index.tolist()
print("unavailable synsets:", missing or "none")
stats.sort_values("multi_token_share", ascending=False)

## ρ for every surviving pair (frames disk-cached via concept_pair)

In [ ]:
rows = []
for a, b, regime, word_level in PAIRS:
    if a in missing or b in missing:
        print(f"skip {a} / {b} (unavailable)")
        continue
    _, _, rho = fur.concept_pair(a, b, MIN_LEMMAS, MAX_TOKENS)
    rows.append(
        {
            "a": a,
            "b": b,
            "regime": regime,
            "word_level": word_level,
            "rho": rho,
            "multi_token_share": float(
                (stats.loc[a, "multi_token_share"] + stats.loc[b, "multi_token_share"]) / 2
            ),
            "has_hub": a in HUB_CONCEPTS or b in HUB_CONCEPTS,
        }
    )

pairs_df = pd.DataFrame(rows).sort_values("rho").reset_index(drop=True)
pairs_df.round(3)

## ρ by semantic regime — the dissociation check

In [ ]:
regimes = pairs_df["regime"].unique().tolist()
fig, ax = plt.subplots(figsize=(9, 4))
for i, regime in enumerate(regimes):
    sub = pairs_df[pairs_df["regime"] == regime]
    ax.scatter([i] * len(sub), sub["rho"], alpha=0.8)
    for _, row in sub.iterrows():
        ax.annotate(
            f"{row['a'].split('.')[0]}/{row['b'].split('.')[0]}",
            (i, row["rho"]),
            fontsize=7,
            xytext=(8, 0),
            textcoords="offset points",
        )
ax.set_xticks(range(len(regimes)), regimes)
ax.set_ylabel("rho")
ax.set_title("concept-pair rho by semantic regime")
plt.tight_layout()
plt.savefig("resources/15_e0_rho_by_regime.png", dpi=150)
plt.show()

print(pairs_df.groupby("regime")["rho"].agg(["mean", "min", "max", "count"]).round(3))

## Composition sanity (RQ3 bookkeeping)

In [ ]:
from frames.representations.concept import Concept

for (a, b), target in COMPOSITION_TARGETS.items():
    if any(s in missing for s in (a, b, target)):
        print(f"skip {a}+{b}~{target}")
        continue
    concept_a = fur.get_concept_cached(a, MIN_LEMMAS, MAX_TOKENS)
    concept_b = fur.get_concept_cached(b, MIN_LEMMAS, MAX_TOKENS)
    concept_t = fur.get_concept_cached(target, MIN_LEMMAS, MAX_TOKENS)
    mean = Concept.average([concept_a, concept_b])
    print(
        f"{a} + {b} ~ {target}:  rho(mean, target) = {mean.rho(concept_t).item():+.3f}"
        f"  [a: {concept_a.rho(concept_t).item():+.3f},"
        f" b: {concept_b.rho(concept_t).item():+.3f}]"
    )

## Select low / medium / high ρ pairs

Deterministic rule: exclude hub-concept pairs; low = lowest ρ, high = highest ρ,
medium = nearest to the median — breaking near-ties (±0.02) toward higher
multi-token share.

In [ ]:
eligible = pairs_df[~pairs_df["has_hub"]].reset_index(drop=True)


def pick(candidates: pd.DataFrame, target_rho: float) -> pd.Series:
    near = candidates[(candidates["rho"] - target_rho).abs() <= 0.02]
    if near.empty:
        near = candidates.iloc[
            [(candidates["rho"] - target_rho).abs().argmin()]
        ]
    return near.sort_values("multi_token_share", ascending=False).iloc[0]


low = pick(eligible, eligible["rho"].min())
high = pick(eligible, eligible["rho"].max())
median_rho = float(eligible["rho"].median())
remaining = eligible[
    ~eligible.apply(lambda r: r["a"] in (low["a"], high["a"]) and r["b"] in (low["b"], high["b"]), axis=1)
]
medium = pick(remaining, median_rho)

selection = pd.DataFrame([low, medium, high], index=["low", "medium", "high"])
assert len({(r["a"], r["b"]) for _, r in selection.iterrows()}) == 3, "pairs must be distinct"
assert low["rho"] < medium["rho"] < high["rho"], "strata must be ordered"
selection.round(3)

In [ ]:
payload = {
    "config": {
        "model_id": fur.model.id,
        "min_lemmas_per_synset": MIN_LEMMAS,
        "max_token_count": MAX_TOKENS,
        "selection_rule": "min/median/max rho, hub-excluded, ties to multi-token share",
    },
    "selected": {
        stratum: {
            "a": row["a"],
            "b": row["b"],
            "rho": float(row["rho"]),
            "regime": row["regime"],
            "word_level": bool(row["word_level"]),
            "multi_token_share": float(row["multi_token_share"]),
        }
        for stratum, row in selection.iterrows()
    },
    "all_pairs": pairs_df.to_dict(orient="records"),
}

out_path = Path("resources/15_e0_selected_pairs.json")
out_path.write_text(json.dumps(payload, indent=2))
print(f"saved {out_path}")
for stratum, info in payload["selected"].items():
    print(f"{stratum:>6}: {info['a']} / {info['b']}  rho={info['rho']:.3f}  ({info['regime']})")